# DM_G3_P0005_transportation_network

## 0. 학습 범위

- Course code: DM
- Gate: G3
- Phase: P0005
- Topic: Ch.5 수송·경유수송·할당·최소비용흐름과 Solver 구조
- Goal: LP를 행렬 수송모형과 node balance 네트워크 모형으로 재표현한다.
- Source basis: `DM_PDF06__04_rag.md`, `DM_PDF02__04_rag.md`
- Output language: Korean
- Code mode: hint_only
- Web grounding: no


## 1. 작성 규칙

- 풀이용 노트북에는 최종 수송계획, 최종 총비용, 정답 계산 결과를 넣지 않는다.
- 각 문제에는 `node_id`와 `source anchor`를 포함한다.
- 행합/열합 수송모형과 node balance 네트워크 모형을 구분한다.
- code cell은 hint_only이며, 답이 드러나는 최적화 코드는 넣지 않는다.


## 2. 채점 기준

- transportation formulation과 Solver mapping: 20점
- balanced/unbalanced/dummy 처리: 15점
- assignment formulation: 15점
- transshipment node balance: 20점
- minimum cost flow와 SUMIF balance: 20점
- sensitivity 연결과 오답진단: 10점


## 3. 네트워크 모델링 기준표

| 모형 | 기본 구조 | 핵심 제약 |
| --- | --- | --- |
| transportation | 공급지 x 수요지 matrix | 행합=공급, 열합=수요 |
| assignment | 공급량=1, 수요량=1인 수송문제 | 행합=1, 열합=1 |
| transshipment | arc flow + 경유지 | 경유지 유입=유출 |
| minimum cost flow | arc table + node balance | 총유출-총유입=의무유출량 |
| Solver | SUMPRODUCT/SUMIF | 목적셀과 제약셀 분리 |


## 4. 문제 세트


### 문제 1. 지역 물류창고 수송계획 — balanced/unbalanced transportation

한 온라인 유통회사가 세 물류센터에서 네 판매지역으로 물량을 보낸다.

공급량: 센터 A=180, 센터 B=220, 센터 C=160  
수요량: 지역1=140, 지역2=130, 지역3=170, 지역4=120

단위 수송비용:

| 공급지/수요지 | 지역1 | 지역2 | 지역3 | 지역4 |
| --- | ---: | ---: | ---: | ---: |
| 센터 A | 6 | 8 | 10 | 9 |
| 센터 B | 9 | 7 | 4 | 5 |
| 센터 C | 8 | 6 | 7 | 6 |

요구:

1. 총공급량과 총수요량을 비교하고 balanced transportation인지 판정하라.
2. decision variable `x_ij`를 정의하라.
3. objective function을 쓰라.
4. 공급지별 행합 제약을 쓰라.
5. 수요지별 열합 제약을 쓰라.
6. nonnegativity 조건을 쓰라.
7. Excel Solver에서 changing cells, target cell, constraint LHS/RHS cells를 설명하라.
8. SUMPRODUCT 목적셀을 어떻게 구성하는지 설명하라.
9. 센터 C 공급량이 120으로 감소하면 dummy supply 또는 dummy demand가 필요한지 판정하라.
10. dummy node 비용을 0으로 둘지 큰 비용으로 둘지 상황별로 설명하라.
11. 수송문제 정수해 성질을 설명하라. 단, 이것이 정수계획을 의미하는 것은 아님을 명시하라.
12. 비용최소화 문제에서 미사용 경로의 reduced cost가 양수이면 무슨 의미인지 설명하라.

- node_id: `n_DM_PDF06.transportation_problem`, `n_DM_PDF06.balanced_transportation`, `n_DM_PDF06.unbalanced_transportation`, `n_DM_PDF06.dummy_supply_or_demand`, `n_DM_PDF06.transportation_solver_model`, `n_DM_PDF06.transportation_integrality`, `n_DM_PDF06.transportation_sensitivity`
- source anchors: `DM_PDF06:p002:L002`, `DM_PDF06:p004:L002`, `DM_PDF06:p005:L002`, `DM_PDF06:p008:L001`, `DM_PDF06:p010:L002`, `DM_PDF06:p010:L013`, `DM_PDF06:p012:L002`, `DM_PDF06:p013:L002`


In [ ]:
# 힌트:
# - 수송문제의 changing cells는 수송량 행렬이다.
# - 목적셀은 SUMPRODUCT(단위비용 행렬, 수송량 행렬)이다.
# - 행합은 공급지별 실제공급량, 열합은 수요지별 실제수요량이다.
# - 경유지에서는 유입량과 유출량을 동시에 고려한다.
# - minimum cost flow의 node balance는 총유출량 - 총유입량 = 의무유출량이다.
# - SUMIF(시작노드, node, 흐름량) - SUMIF(종료노드, node, 흐름량)을 사용한다.


### 내 답안

- balanced 판정:
- 변수 정의:
- 목적함수:
- 행합 제약:
- 열합 제약:
- Solver mapping:
- dummy 처리:
- 정수해 성질:
- reduced cost 해석:
- node_id/source anchors:


### 문제 2. 장비-작업 할당문제 — assignment as transportation

네 장비를 네 작업에 배정한다. 각 장비는 정확히 하나의 작업만 수행하고, 각 작업도 정확히 하나의 장비에만 배정된다. 비용은 준비시간이다.

| 장비/작업 | 작업1 | 작업2 | 작업3 | 작업4 |
| --- | ---: | ---: | ---: | ---: |
| 장비1 | 11 | 8 | 7 | 10 |
| 장비2 | 9 | 12 | 6 | 7 |
| 장비3 | 8 | 7 | 10 | 6 |
| 장비4 | 6 | 9 | 8 | 11 |

요구:

1. `x_ij`를 정의하라. `x_ij`의 의미를 0/1로 설명하라.
2. objective function을 쓰라.
3. 장비별 행합 제약을 쓰라.
4. 작업별 열합 제약을 쓰라.
5. 왜 공급량=1, 수요량=1인 특별한 수송문제인지 설명하라.
6. Solver에서 changing cells는 어떤 4x4 matrix인지 설명하라.
7. 행합=1, 열합=1 제약을 Solver에서 어떻게 입력하는지 설명하라.
8. binary constraint 명시 여부를 수송문제 정수해 성질과 의미상 0/1 해석으로 구분하라.
9. “행합 <= 1이면 충분하고 작업별 제약은 없어도 된다”는 오답을 진단하라.
10. assignment와 set partitioning의 연결을 한 문장으로 설명하라.

- node_id: `n_DM_PDF06.assignment_problem`, `n_DM_PDF06.transportation_integrality`
- source anchors: `DM_PDF06:p003:L009`, `DM_PDF06:p015:L002`, `DM_PDF06:p010:L013`


In [ ]:
# 힌트:
# - 수송문제의 changing cells는 수송량 행렬이다.
# - 목적셀은 SUMPRODUCT(단위비용 행렬, 수송량 행렬)이다.
# - 행합은 공급지별 실제공급량, 열합은 수요지별 실제수요량이다.
# - 경유지에서는 유입량과 유출량을 동시에 고려한다.
# - minimum cost flow의 node balance는 총유출량 - 총유입량 = 의무유출량이다.
# - SUMIF(시작노드, node, 흐름량) - SUMIF(종료노드, node, 흐름량)을 사용한다.


### 내 답안

- 변수 정의:
- 목적함수:
- 행합/열합 제약:
- 수송문제 특수형 설명:
- Solver mapping:
- binary/정수해 성질 구분:
- 오답 진단:
- node_id/source anchors:


### 문제 3. 냉장식품 경유수송 — transshipment problem

두 공장에서 두 중간창고를 거쳐 세 매장으로 냉장식품을 보낸다.

공급량: 공장1=150, 공장2=170  
수요량: 매장5=90, 매장6=110, 매장7=120  
중간창고: 창고3, 창고4

허용 경로와 단위비용:

| 시작노드 | 종료노드 | 단위비용 |
| ---: | ---: | ---: |
| 1 | 3 | 4 |
| 1 | 4 | 6 |
| 2 | 3 | 5 |
| 2 | 4 | 3 |
| 3 | 5 | 7 |
| 3 | 6 | 4 |
| 3 | 7 | 8 |
| 4 | 5 | 6 |
| 4 | 6 | 5 |
| 4 | 7 | 4 |

요구:

1. arc flow variable `x_ij`를 정의하라.
2. objective function을 arc table 기반으로 쓰라.
3. 공장 1, 공장 2의 node balance를 쓰라.
4. 창고 3, 창고 4의 node balance를 쓰라.
5. 매장 5, 6, 7의 node balance를 쓰라.
6. 창고는 왜 공급지이면서 동시에 수요지로 처리되는지 설명하라.
7. 단순 transportation matrix와 transshipment node balance의 차이를 설명하라.
8. Solver에서 changing cells는 무엇인가?
9. SUMPRODUCT 목적셀을 어떻게 구성하는가?
10. 창고 3의 순수유출량을 SUMIF 구조로 쓰라.
11. 창고 4의 순수유출량을 SUMIF 구조로 쓰라.
12. “창고는 최종 수요가 없으므로 제약식을 쓰지 않아도 된다”는 오답을 진단하라.

- node_id: `n_DM_PDF06.transshipment_problem`, `n_DM_PDF06.transshipment_balance`, `n_DM_PDF06.network_topology_table`, `n_DM_PDF06.sumif_node_balance`
- source anchors: `DM_PDF06:p003:L002`, `DM_PDF06:p017:L002`, `DM_PDF06:p019:L002`, `DM_PDF06:p030:L001`, `DM_PDF06:p031:L002`, `DM_PDF06:p032:L002`


In [ ]:
# 힌트:
# - 수송문제의 changing cells는 수송량 행렬이다.
# - 목적셀은 SUMPRODUCT(단위비용 행렬, 수송량 행렬)이다.
# - 행합은 공급지별 실제공급량, 열합은 수요지별 실제수요량이다.
# - 경유지에서는 유입량과 유출량을 동시에 고려한다.
# - minimum cost flow의 node balance는 총유출량 - 총유입량 = 의무유출량이다.
# - SUMIF(시작노드, node, 흐름량) - SUMIF(종료노드, node, 흐름량)을 사용한다.


### 내 답안

- arc 변수 정의:
- 목적함수:
- 공장 balance:
- 창고 balance:
- 매장 balance:
- matrix vs node balance:
- SUMIF 구조:
- 오답 진단:
- node_id/source anchors:


### 문제 4. 도시 간 최소비용흐름 — minimum cost flow with capacity

다음 네트워크에서 arc 흐름량을 결정해 총비용을 최소화한다.

노드별 의무유출량: 1=+35, 2=+25, 3=0, 4=-20, 5=-40

| 시작노드 | 종료노드 | 단위비용 | 흐름용량 |
| ---: | ---: | ---: | ---: |
| 1 | 3 | 5 | 30 |
| 1 | 4 | 8 | 20 |
| 2 | 3 | 4 | 25 |
| 2 | 5 | 7 | 30 |
| 3 | 4 | 3 | 25 |
| 3 | 5 | 6 | 30 |
| 4 | 5 | 2 | 20 |

요구:

1. arc별 changing cells를 정의하라.
2. objective function을 쓰라.
3. arc capacity constraints를 쓰라.
4. 각 노드의 node balance를 `총유출량 - 총유입량 = 의무유출량`으로 쓰라.
5. 노드 3의 balance를 수식으로 쓰라.
6. 노드 4의 balance를 수식으로 쓰라.
7. SUMIF를 사용한 node balance 계산 구조를 설명하라.
8. supply node, demand node, transshipment node를 구분하라.
9. transportation problem과 minimum cost flow의 차이를 설명하라.
10. Solver에서 target cell, changing cells, constraint cells, RHS cells를 설명하라.
11. 수송문제 sensitivity와 minimum cost flow sensitivity를 연결해 설명하라.
12. “모든 노드는 유출량이 유입량보다 커야 한다”는 오답을 진단하라.

- node_id: `n_DM_PDF06.minimum_cost_flow`, `n_DM_PDF06.network_topology_table`, `n_DM_PDF06.sumif_node_balance`, `n_DM_PDF02.shadow_price`, `n_DM_PDF02.reduced_cost`
- source anchors: `DM_PDF06:p022:L002`, `DM_PDF06:p025:L002`, `DM_PDF06:p026:L002`, `DM_PDF06:p030:L001`, `DM_PDF06:p031:L002`, `DM_PDF06:p032:L002`


In [ ]:
# 힌트:
# - 수송문제의 changing cells는 수송량 행렬이다.
# - 목적셀은 SUMPRODUCT(단위비용 행렬, 수송량 행렬)이다.
# - 행합은 공급지별 실제공급량, 열합은 수요지별 실제수요량이다.
# - 경유지에서는 유입량과 유출량을 동시에 고려한다.
# - minimum cost flow의 node balance는 총유출량 - 총유입량 = 의무유출량이다.
# - SUMIF(시작노드, node, 흐름량) - SUMIF(종료노드, node, 흐름량)을 사용한다.


### 내 답안

- arc changing cells:
- objective:
- capacity constraints:
- node balance:
- SUMIF 구조:
- node 유형:
- Solver mapping:
- sensitivity 연결:
- 오답 진단:
- node_id/source anchors:


### 문제 5. Phase 4-5 연결 문제: 수송문제 민감도 해석

문제 1의 수송계획을 Solver로 풀었다고 하자. 민감도 보고서 일부가 다음과 같이 주어졌다.

Variable Cells 일부:

| 경로 | Final Value | Reduced Cost | Unit Cost | Allowable Decrease |
| --- | ---: | ---: | ---: | ---: |
| A→지역3 | 0 | 4 | 10 | 4 |
| C→지역1 | 0 | 3 | 8 | 3 |

Constraints 일부:

| 제약 | Shadow Price | RHS |
| --- | ---: | ---: |
| 센터 B 공급량 | -2 | 220 |
| 지역 4 수요량 | 5 | 120 |

요구:

1. 비용최소화 문제에서 미사용 경로 A→지역3의 reduced cost = 4는 무슨 뜻인지 설명하라.
2. A→지역3의 단위비용이 10에서 7로 낮아지면 현재 수송계획이 유지되는지 판정하라.
3. 센터 B 공급량이 10 증가하면 총비용이 어떻게 변하는지 계산하라. 단, 허용범위 안이라고 가정한다.
4. 지역 4 수요량이 15 증가하면 총비용이 어떻게 변하는지 계산하라. 단, 허용범위 안이라고 가정한다.
5. “센터 B의 shadow price가 -2이므로 공급량이 늘면 비용이 줄어든다”는 말을 현실 언어로 해석하라.
6. reduced cost와 shadow price를 다시 변수 질문/제약 질문으로 구분하라.

- node_id: `n_DM_PDF06.transportation_sensitivity`, `n_DM_PDF02.reduced_cost`, `n_DM_PDF02.shadow_price`
- source anchors: `DM_PDF06:p012:L002`, `DM_PDF06:p013:L002`, `DM_PDF06:p014:L002`, `DM_PDF02:p032:L002`


In [ ]:
# 힌트:
# - 수송문제의 changing cells는 수송량 행렬이다.
# - 목적셀은 SUMPRODUCT(단위비용 행렬, 수송량 행렬)이다.
# - 행합은 공급지별 실제공급량, 열합은 수요지별 실제수요량이다.
# - 경유지에서는 유입량과 유출량을 동시에 고려한다.
# - minimum cost flow의 node balance는 총유출량 - 총유입량 = 의무유출량이다.
# - SUMIF(시작노드, node, 흐름량) - SUMIF(종료노드, node, 흐름량)을 사용한다.


### 내 답안

- reduced cost 해석:
- 단위비용 7 판정:
- 공급량 증가 비용 변화:
- 수요량 증가 비용 변화:
- shadow price 현실 언어:
- 변수 질문/제약 질문 구분:
- node_id/source anchors:
